# EEE 457 Transmissão de Energia Elétrica
## Escola Politécnica
## Universidade Federal do Rio de Janeiro
### Antonio C. S. Lima
Outubro 2025
### Desequilibrio em circuito com modelo de linha curta 

In [1]:
# carrega as bibliotecas

import numpy as np
from numpy import pi, sqrt, log, exp, diag, eye, kron
from numpy.linalg import inv, eig
from scipy.special import iv, kv
import matplotlib.pyplot as plt
import pandas as pd
from typing import Iterable, Tuple, Optional, Union
import warnings 
import os
import line_cable_param as lcp

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.size": 14,
    "font.sans-serif": "Helvetica",
})

In [2]:
# constantes importantes
µ0 = 4e-7 * pi  # permeabilidade do vácuo (H/m)
ε0 = 8.854187817e-12  # permissividade do vácuo (F/m)
c0 = 1 / sqrt(µ0 * ε0)  # velocidade da luz no vácuo (m/s)
η0 = sqrt(µ0 / ε0)  # impedância característica do vácuo (Ohms)
freq = 60  # frequência de operação (Hz)
omega = 2 * pi * freq  # frequência angular (rad/s)
a = np.exp( 2j * pi / 3)  #  1/_ 120 graus para o calculo dos parâmetros de sequência 

In [3]:
# primeira configuração 
# Circuito 345 kV (345CSH1) com dois condutores Tern
s = 0.4572  # m
f = 19.1  # m flecha dos condutores
fpr = 15.24  # m flecha dos para-raios
xc = np.array([-8.5 - s/2,  -8.5 + s/2, - s/2, s/2, 8.5 - s/2, 8.5 + s/2, -6.25, 6.25] ) # m
yc = np.array([28.4, 28.4, 29.25, 29.25, 28.4, 28.4, 35.9, 35.9 ])  # m
fc = np.concat( [6 * [f], 2* [fpr] ] )  # m flechas dos condutores 
yc = np.array(yc) - 2/3 * fc  # altura media dos condutores 
rho_s = 1e3                   # restividade do solo em  (Ohm . m)
ri = (6.75e-3)/2              # raio interno dos condutores de fase (m)
rf = (27.01e-3)/2             # raio externo dos condutores de fase (m)
rdc = 0.0734e-3               # resistência DC dos condutores de fase (Ohm/m)
nb = 2
npr = 2
rpr = 4.57e-3               # raio dos para-raios (m) considerando 3/8" EHS
rdc_pr =  4.190e-3           # resistência DC dos para-raios (Ohm/m)

In [4]:
# czyl_overhead_bundled(omega, x, y, sigma_s, rdc, rf, rint, npr, rdcpr, rpr, nb):
Z, Y = lcp.czyl_overhead_bundled(omega = omega, 
                                 x = xc,
                                 y = yc,
                                 sigma_s = 1/rho_s,
                                 rdc = rdc, 
                                 rf = rf,
                                 rint = ri, 
                                 npr = npr, 
                                 rdcpr = rdc_pr, 
                                 rpr = rpr,
                                 nb = nb)

In [5]:
print("Impedance Matrix (Ohm/km):")
print(Z * 1000)

Impedance Matrix (Ohm/km):
[[0.13502452+0.75447764j 0.09921839+0.39127967j 0.09741758+0.34096048j]
 [0.09921839+0.39127967j 0.13821513+0.75151352j 0.09921839+0.39127967j]
 [0.09741758+0.34096048j 0.09921839+0.39127967j 0.13502452+0.75447764j]]


In [6]:
print("Admitance (S/km):")
print(Y * 1000)

Admitance (S/km):
[[3.e-09+3.80677820e-06j 0.e+00-6.77018964e-07j 0.e+00-2.13597971e-07j]
 [0.e+00-6.77018964e-07j 3.e-09+3.93105230e-06j 0.e+00-6.77018964e-07j]
 [0.e+00-2.13597971e-07j 0.e+00-6.77018964e-07j 3.e-09+3.80677820e-06j]]


### Empregando o modelo de linha curta e supondo a carga equilibrada

In [7]:
# circuito com comprimento de 90km e carga equilibrada
# calculo  do desequilibrio na fonte
Vr = 345e3/np.sqrt(3) * np.array([1, a ** 2, a])
Ir = Vr/zc
Vs= np.sqrt(3)* (Vr + Z * 90.e3 @ Ir) 
v012 = 1/345e3 * A_inv @ Vs
np.abs(v012)

NameError: name 'zc' is not defined

In [ ]:
para esse cenário calcule a relação v2/v1 e v0/v1

### Empregando o modelo de linha média e supondo a carga equilibrada

In [ ]:
# considerando 180km para o comprimento do circuito 
# matriz de rotação do circuito
r = np.array([[0,1,0],[0,0,1],[1,0,0]])
rinv = np.linalg.inv(r)

# impedancia do primeiro trecho
Zi = Z * 60e3
Zii = rinv @ Zi @ r
Ziii= rinv @ Zii @ r
# impedancia após a transposição de fases
Z1 = 1/3 *(Zi + Zii + Ziii)

# admitancia do primeiro trecho
Yi = Y * 60e3
Yii = rinv @ Yi @ r
Yiii= rinv @ Yii @ r
# admitancia após a transposição de fases
Y1 = 1/3 *(Yi + Yii + Yiii)
# atualizando o parâmetro do quadripolo do circuito
Aq = np.eye(3) + 0.5 * (Z1 @ Y1)
Vs1 =np.sqrt(3)*( Aq @ Vr + Z1 @ Ir)
v012 = 1/345e3 * A_inv @ Vs1
np.abs(v012)

#### Compare os resultados acima, consegue identificar as causas das diferenças?

### Suponha agora um circuito com 240 km de comprimento
obtenha as matrizes de impedância por unidade de comprimento, calcule as matrizes equivalentes e obtenha os níveis de desequilibrio
repita o procedimento, mas representado cada trecho do circuito de forma independente:
represente 1 quadripolo para cada trecho do circuito, conecte os trechos não transpostos através da matriz de rotação e obtenha as tensões e correntes no terminal receptor, supondo a carga equilibrada, compare com o resultado anterior